In [18]:
import pickle
import pandas as pd
from tqdm import tqdm

In [ ]:
import os
# Change the working directory to the SageNet repository root
# This is necessary to ensure that the relative paths work correctly
root_dir = "root_directory_path"  # Replace with the actual root directory path 
os.chdir(root_dir)

In [ ]:
PATH_COLLECTION_DATA = 'data\initial_dataset.pkl'
df = pd.read_pickle(PATH_COLLECTION_DATA)

In [5]:
df.columns

Index(['cord_uid', 'source_x', 'title', 'doi', 'pmcid', 'pubmed_id', 'license',
       'abstract', 'publish_time', 'authors', 'journal', 'mag_id',
       'who_covidence_id', 'arxiv_id', 'label', 'time', 'timet'],
      dtype='object')

In [9]:
records = []

for _, row in df.iterrows():

    cord_uid = str(row["cord_uid"])
    title = str(row.get("title", ""))
    abstract = str(row.get("abstract", ""))
    
    # Append to records list
    records.append({
        "id": cord_uid,
        "chunk_text": title + " " + abstract,
    })

In [ ]:
#pinecone setup

import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv

env_path = ".env"
load_dotenv(dotenv_path=env_path)

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "sagenet-serverless-index"

# Initialize index client
index = pc.Index(name=index_name)

# View index stats
print(index.describe_index_stats())

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'default': {'vector_count': 7718}},
 'total_vector_count': 7718,
 'vector_type': 'dense'}


In [ ]:
# import time

# batch_size = 96 #max batchsize for serverless index
# sleep_seconds = 10  # for see rate limit errors

# total_records = len(records)
# print(f"Starting upsertion of {total_records} vectors in batches of {batch_size}...")

# for i in range(0, total_records, batch_size):
#     batch = records[i:i + batch_size]
    
#     try:
#         index.upsert_records(records=batch, namespace="default")
#         print(f"Batch {i//batch_size + 1}: Upserted {len(batch)} vectors")
#     except Exception as e:
#         print(f"Error on batch {i//batch_size + 1}: {e}")
#         print("Sleeping 5 seconds and retrying...")
#         time.sleep(20)
#         try:
#             index.upsert_records(records=batch, namespace="default")
#             print(f"Batch {i//batch_size + 1}: Upserted on retry")
#         except Exception as e2:
#             print(f"Failed again: {e2}")
            
#     # Rate limit safety pause
#     time.sleep(sleep_seconds)

# print("All batches upserted!")

Starting upsertion of 7718 vectors in batches of 96...
Batch 1: Upserted 96 vectors
Batch 2: Upserted 96 vectors
Batch 3: Upserted 96 vectors
Batch 4: Upserted 96 vectors
Batch 5: Upserted 96 vectors
Batch 6: Upserted 96 vectors
Batch 7: Upserted 96 vectors
Batch 8: Upserted 96 vectors
Batch 9: Upserted 96 vectors
Batch 10: Upserted 96 vectors
Batch 11: Upserted 96 vectors
Batch 12: Upserted 96 vectors
Batch 13: Upserted 96 vectors
Batch 14: Upserted 96 vectors
Batch 15: Upserted 96 vectors
Batch 16: Upserted 96 vectors
Batch 17: Upserted 96 vectors
Batch 18: Upserted 96 vectors
Batch 19: Upserted 96 vectors
Batch 20: Upserted 96 vectors
Batch 21: Upserted 96 vectors
Batch 22: Upserted 96 vectors
Batch 23: Upserted 96 vectors
Batch 24: Upserted 96 vectors
Batch 25: Upserted 96 vectors
Batch 26: Upserted 96 vectors
Batch 27: Upserted 96 vectors
Batch 28: Upserted 96 vectors
Batch 29: Upserted 96 vectors
Batch 30: Upserted 96 vectors
Batch 31: Upserted 96 vectors
Batch 32: Upserted 96 ve

In [ ]:
#query testing

query = "How has the COVID-19 pandemic affected mental health?"

results = index.search(
    namespace="default",
    query={
        "inputs": {"text": query},
        "top_k": 3
    }
)

print(results)

{'result': {'hits': [{'_id': 'lfvvfzi5',
                      '_score': 0.6267624497413635,
                      'fields': {'chunk_text': 'Prevalence of mental health '
                                               'problems during the COVID-19 '
                                               'pandemic: A systematic review '
                                               'and meta-analysis BACKGROUND: '
                                               ': The global COVID-19 pandemic '
                                               'has generated major mental and '
                                               'psychological health problems '
                                               'worldwide. We conducted a '
                                               'meta-analysis to assess the '
                                               'prevalence of depression, '
                                               'anxiety, distress, and '
                                             

In [22]:
results['result']['hits'][0]['_id']  # Example to access the first result's ID

'lfvvfzi5'

In [15]:
results['result']['hits'][1]['fields']['chunk_text']

'Mental Health, Substance Use, and Suicidal Ideation During the COVID-19 Pandemic — United States, June 24–30, 2020 The coronavirus disease 2019 (COVID-19) pandemic has been associated with mental health challenges related to the morbidity and mortality caused by the disease and to mitigation activities, including the impact of physical distancing and stay-at-home orders.* Symptoms of anxiety disorder and depressive disorder increased considerably in the United States during April-June of 2020, compared with the same period in 2019 (1,2). To assess mental health, substance use, and suicidal ideation during the pandemic, representative panel surveys were conducted among adults aged ≥18 years across the United States during June 24-30, 2020. Overall, 40.9% of respondents reported at least one adverse mental or behavioral health condition, including symptoms of anxiety disorder or depressive disorder (30.9%), symptoms of a trauma- and stressor-related disorder (TSRD) related to the pandem

In [17]:
results['result']['hits'][2]['fields']['chunk_text']

'Mental health during the COVID-19 pandemic in two longitudinal UK population cohorts Background: The impact of COVID-19 on mental health is unclear. Evidence from longitudinal studies with pre pandemic data are needed to address (1) how mental health has changed from pre-pandemic levels to during the COVID-19 pandemic and (2), whether there are groups at greater risk of poorer mental health during the pandemic? Methods: We used data from COVID-19 surveys (completed through April/May 2020), nested within two large longitudinal population cohorts with harmonised measures of mental health: two generations of the Avon Longitudinal Study of Parents and Children (ALPSAC): the index generation ALSPAC-G1 (n= 2850, mean age 28) and the parents generation ALSPAC-G0 (n= 3720, mean age = 59) and Generation Scotland: Scottish Family Health Study (GS, (n= 4233, mean age = 59), both with validated pre-pandemic measures of mental health and baseline factors. To answer question 1, we used ALSPAC-G1, w

In [26]:
topk_corduids=[]
for i in range(len(results['result']['hits'])):
    topk_corduids.append(results['result']['hits'][i]['_id'])

In [27]:
topk_corduids

['lfvvfzi5', 'e5tey69t', 'a3lk2q02']